In [1]:
import numpy as np
import pandas as pd

from exerpy import ExergyAnalysis
from exerpy.analyses import _load_json

In [2]:
def diff_between_simulators(testcase):
    simulator_results = []
    for path in testcase.values():
        contents = _load_json(path)
        if "settings" not in contents:
            contents["settings"] = {}

        simulator_results += [
            ExergyAnalysis.from_json(path, **contents["settings"])
        ]

    sim1 = simulator_results[0]
    sim2 = simulator_results[1]

    columns = ["m", "p", "T"]
    if sim1.chemExLib is not None:
        columns.append("e_CH")
    if sim1.split_physical_exergy:
        columns.append("e_M")
        columns.append("e_T")
    else:
        columns.append("e_PH")

    df_sim1 = pd.DataFrame.from_dict(
        sim1._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")
    df_sim2 = pd.DataFrame.from_dict(
        sim2._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")

    overlapping_index = list(
        set(df_sim1.index.tolist()) & set(df_sim2.index.tolist())
    )
    df_sim1 = df_sim1.loc[overlapping_index].round(6)
    df_sim2 = df_sim2.loc[overlapping_index].round(6)

    # inf means that sim2 has 0 value, comparison does not make sense there
    # and sometimes there seem to be NaN values in the dataframes, those are
    # removed as well
    diff = ((df_sim1 - df_sim2) / df_sim2).abs().replace(np.inf, 0).fillna(0)
    # exergies near the ambient state are close to zero, so differences below 1 J/kg are round-off
    exergies = [column for column in columns if column.startswith("e_")]
    diff[exergies] = diff[exergies].mask((df_sim1 - df_sim2)[exergies].abs() < 1.0, 0)
    return diff

In [3]:
testcase = {
    "ebsilon": "cgam_ebs.json",
    "tespy": "cgam_tespy.json"
}
diff_between_simulators(testcase)


,m,p,T,e_CH,e_M,e_T
4,0.000095,0.000036,1.973684e-08,0.002036,0.000376,0.000006
6,0.000095,0.000296,6.313108e-04,0.002036,0.005740,0.001019
3,0.000078,0.000052,0.000000e+00,0.000000,0.000484,0.000578
7,0.000095,0.000000,8.631374e-04,0.002036,0.000000,0.003687
5,0.000095,0.000268,3.165053e-04,0.002036,0.003242,0.000089
9,0.000000,0.000000,2.059617e-09,0.000000,0.000000,0.000000
10,0.001046,0.000000,0.000000e+00,0.000000,0.000193,0.000000
8P,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
2,0.000078,0.000000,3.463453e-04,0.000000,0.000527,0.000449
6P,0.000095,0.000177,8.023187e-04,0.002036,0.006924,0.002191


In [4]:
testcase = {
    "tespy": "cgam_tespy.json",
    "aspen": "cgam_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_M,e_T
4,0.000440,0.000036,0.000000e+00,0.0,0.000603,0.000465
6,0.000464,0.000296,5.221679e-05,0.0,0.006006,0.000027
3,0.000440,0.000052,0.000000e+00,0.0,0.000532,0.001059
7,0.000464,0.000000,3.549676e-04,0.0,0.000000,0.001786
5,0.000464,0.000268,1.269831e-05,0.0,0.003489,0.000129
9,0.000000,0.000000,0.000000e+00,0.0,0.000000,0.000000
10,0.000403,0.000000,0.000000e+00,0.0,0.000000,0.000000
8P,0.000000,0.000000,4.781872e-07,0.0,0.000000,0.000000
2,0.000440,0.000000,1.931286e-04,0.0,0.000524,0.002304
6P,0.000464,0.000177,2.449095e-04,0.0,0.006652,0.000692


In [5]:
testcase = {
    "ebsilon": "cgam_ebs.json",
    "aspen": "cgam_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_CH,e_M,e_T
4,0.000344,0.000000e+00,1.973684e-08,0.002121,0.000979,0.000470
6,0.000369,3.752346e-11,5.791269e-04,0.002180,0.000000,0.001045
3,0.000362,0.000000e+00,0.000000e+00,0.000000,0.001016,0.000482
7,0.000369,0.000000e+00,5.084762e-04,0.002180,0.000000,0.001907
5,0.000369,3.639674e-11,3.038111e-04,0.002180,0.000236,0.000218
9,0.000000,0.000000e+00,2.059617e-09,0.000000,0.000000,0.000000
10,0.000644,0.000000e+00,0.000000e+00,0.000000,0.000191,0.000000
8P,0.000000,0.000000e+00,4.781872e-07,0.000000,0.000000,0.000000
2,0.000362,0.000000e+00,5.394071e-04,0.000000,0.001052,0.002752
6P,0.000369,3.848005e-11,5.576057e-04,0.002180,0.000000,0.001501


In [6]:
# Component results stored by the example scripts cgam_ebs.py, cgam_tespy.py and cgam_aspen.py
results = {
    simulator: pd.read_csv(f"cgam_components_{simulator.lower()}.csv", index_col=0).set_index("Component")
    for simulator in ("Ebsilon", "TESPy", "Aspen")
}

# Components the simulators model differently are compared as one group
groups = {
    "COMB": "CC",  # Aspen name of the combustion chamber
    "DRUM": "EV",  # TESPy models the drum separately
}

# Exergy destruction in kW of the components found in at least two models, and the deviations from Ebsilon
E_D = (
    pd.DataFrame({simulator: df["E_D [kW]"] for simulator, df in results.items()})
    .drop("TOT")
    .rename(index=groups)
    .groupby(level=0)
    .sum(min_count=1)
    .dropna(thresh=2)
)
reference = E_D["Ebsilon"].where(E_D["Ebsilon"].abs() > 1e-3)  # no relative deviation below 1 W
for simulator in ("TESPy", "Aspen"):
    E_D[f"{simulator} vs Ebsilon [%]"] = (E_D[simulator] / reference - 1) * 100
E_D.sort_values("Ebsilon", ascending=False).round(2)

,Ebsilon,TESPy,Aspen,TESPy vs Ebsilon [%],Aspen vs Ebsilon [%]
Component,,,,,
CC,25332.37,25397.79,25376.43,0.26,0.17
EV,4587.75,4600.59,4594.86,0.28,0.16
EXP,2996.18,2995.01,2994.61,-0.04,-0.05
APH,2558.61,2557.50,2557.42,-0.04,-0.05
AC,2102.24,2101.22,2100.76,-0.05,-0.07
PH,1912.09,1915.95,1915.39,0.20,0.17
GEN,0.00,NaN,0.00,NaN,NaN


In [7]:
# Results of the whole plant
columns = ["E_F [kW]", "E_P [kW]", "E_D [kW]", "E_L [kW]", "epsilon [%]"]
pd.DataFrame({simulator: df.loc["TOT", columns] for simulator, df in results.items()}).round(3)

,Ebsilon,TESPy,Aspen
E_F [kW],85106.352,85195.392,85161.074
E_P [kW],42753.648,42753.646,42753.638
E_D [kW],39489.245,39568.062,39539.457
E_L [kW],2863.459,2873.683,2867.979
epsilon [%],50.236,50.183,50.203
